# Spam classification
Curated from Carl James Poopalaretnam's PS3 coursework. Split raw rows before fitting a scaler inside the training pipeline. This uses a stratified 80/20 split (seed 42), so results are not directly comparable with the original unstratified, leaked preprocessing.
UCI Spambase is an academic historical dataset; scores do not establish performance on modern email. Data instructions are in data/README.md.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/spam_classification.ipynb").exists())
data_root = Path(os.environ.get("ML_DATA_DIR") or root / "data/raw").expanduser()
if not data_root.is_absolute(): data_root = root / data_root
data = pd.read_csv(data_root / "spambase/spambase.data", header=None)
if data.shape[1] != 58: raise ValueError("Expected 57 input features and one label")
X, y = data.iloc[:, :-1], data.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear", max_iter=1000, random_state=42))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f"Rows: {len(data)}; train: {len(X_train)}; test: {len(X_test)}")
print(f"Accuracy: {accuracy_score(y_test, prediction):.6f}")
print(classification_report(y_test, prediction, digits=4))
np.testing.assert_allclose(model.named_steps["standardscaler"].mean_, X_train.mean(axis=0))
